In [77]:
# 4.1 - Thư viện và đường dẫn dữ liệu
from pathlib import Path
import numpy as np
import pandas as pd

# Tìm thư mục dữ liệu trong dự án hoặc thư mục hiện tại
DATA_DIR = None
for folder in [Path.cwd(), *Path.cwd().parents]:
    for candidate in [folder / "data" / "raw", folder / "data", folder]:
        if (candidate / "fact_transaction.csv").exists():
            DATA_DIR = candidate
            break
    if DATA_DIR is not None:
        break

if DATA_DIR is None:
    raise FileNotFoundError("Không tìm thấy fact_transaction.csv. Hãy kiểm tra thư mục data/raw")

def read(name, columns=None):
    return pd.read_csv(DATA_DIR / f"{name}.csv", usecols=columns)

print("Thư mục dữ liệu:", DATA_DIR.resolve())

Thư mục dữ liệu: D:\banking-fraud-detection\data\raw


In [78]:
# 4.2 - Đọc giao dịch và các bảng liên quan
tx = read("fact_transaction")
customers = read("dim_customer", ["customer_key", "date_of_birth"])
accounts = read("dim_account", ["account_key", "open_date"])
branches = read("dim_branch", ["branch_key", "open_date"])
types = read("dim_transaction_type", ["transaction_type_key", "type_code"])
logins = read("fact_login_event", ["customer_key", "status", "login_at"])
labels = read("fact_fraud_label_event", ["transaction_id", "label", "label_source", "labeled_at"])
print("Số giao dịch:", len(tx))

Số giao dịch: 30000


In [79]:
# 4.3 - Kiểm tra khóa chính
assert tx["transaction_id"].notna().all() and tx["transaction_id"].is_unique, "Lỗi khóa giao dịch"
assert customers["customer_key"].notna().all() and customers["customer_key"].is_unique, "Lỗi khóa khách hàng"
assert accounts["account_key"].notna().all() and accounts["account_key"].is_unique, "Lỗi khóa tài khoản"
assert branches["branch_key"].notna().all() and branches["branch_key"].is_unique, "Lỗi khóa chi nhánh"
assert types["transaction_type_key"].notna().all() and types["transaction_type_key"].is_unique, "Lỗi khóa loại giao dịch"
print("Kiểm tra khóa chính: ĐẠT")

Kiểm tra khóa chính: ĐẠT


In [80]:
# 4.4 - Chuẩn hóa kiểu ngày giờ và kiểm tra số tiền
# Ngày giờ phải được chuyển đổi trước khi ghép bảng và tạo đặc trưng
tx["transaction_at"] = pd.to_datetime(tx["transaction_at"])
accounts["open_date"] = pd.to_datetime(accounts["open_date"])
branches["open_date"] = pd.to_datetime(branches["open_date"])
customers["date_of_birth"] = pd.to_datetime(customers["date_of_birth"])
logins["login_at"] = pd.to_datetime(logins["login_at"])
labels["labeled_at"] = pd.to_datetime(labels["labeled_at"])

assert tx["transaction_at"].notna().all(), "Thiếu thời gian giao dịch"
assert tx["amount"].notna().all(), "Thiếu số tiền giao dịch"
assert (tx["amount"] >= 0).all(), "Có số tiền giao dịch âm"
print("Thời gian và số tiền: ĐẠT")

Thời gian và số tiền: ĐẠT


In [81]:
# 4.5 - Kiểm tra liên kết giữa các bảng
assert tx["customer_key"].isin(customers["customer_key"]).all(), "Không tìm thấy khách hàng"
assert tx["source_account_key"].isin(accounts["account_key"]).all(), "Không tìm thấy tài khoản nguồn"
assert tx["branch_key"].isin(branches["branch_key"]).all(), "Không tìm thấy chi nhánh"
assert tx["transaction_type_key"].isin(types["transaction_type_key"]).all(), "Không tìm thấy loại giao dịch"
print("Kiểm tra liên kết bảng: ĐẠT")

Kiểm tra liên kết bảng: ĐẠT


In [82]:
# 4.6 - Ghép dữ liệu giao dịch với các bảng chiều
# Không đổi tên trực tiếp trong bảng gốc để tránh lỗi khi chạy lại cell
account_info = accounts.rename(columns={"account_key": "source_account_key", "open_date": "account_open_date"})
branch_info = branches.rename(columns={"open_date": "branch_open_date"})

modeling_df = tx.copy()
modeling_df = modeling_df.merge(customers, on="customer_key", how="left", validate="many_to_one")
modeling_df = modeling_df.merge(account_info, on="source_account_key", how="left", validate="many_to_one")
modeling_df = modeling_df.merge(branch_info, on="branch_key", how="left", validate="many_to_one")
modeling_df = modeling_df.merge(types, on="transaction_type_key", how="left", validate="many_to_one")

assert len(modeling_df) == len(tx), "Số giao dịch thay đổi sau ghép"
assert modeling_df["transaction_id"].is_unique, "Trùng mã giao dịch sau ghép"
print("Kích thước sau ghép:", modeling_df.shape)

Kích thước sau ghép: (30000, 20)


In [83]:
# 4.7 - Tạo đặc trưng thời gian
modeling_df["transaction_at"] = pd.to_datetime(modeling_df["transaction_at"])
modeling_df["hour"] = modeling_df["transaction_at"].dt.hour
modeling_df["day_of_week"] = ((modeling_df["transaction_at"].dt.day_of_week + 1) % 7) + 1
modeling_df["is_night_transaction"] = (
    (modeling_df["hour"] >= 22) | (modeling_df["hour"] < 6)
).astype(int)

# Quy ước: Chủ nhật = 1, thứ bảy = 7
print("Đã tạo đặc trưng thời gian")

Đã tạo đặc trưng thời gian


In [84]:
# 4.8 - Tính tuổi tài khoản bằng số ngày nguyên
modeling_df["account_age_days"] = (
    modeling_df["transaction_at"].dt.normalize()
    - modeling_df["account_open_date"].dt.normalize()
).dt.days
assert modeling_df["account_age_days"].notna().all(), "Thiếu ngày mở tài khoản"
assert (modeling_df["account_age_days"] >= 0).all(), "Ngày mở tài khoản không hợp lệ"
print("Đã tính account_age_days ")

Đã tính account_age_days 


In [85]:
# 4.9 - Kiểm tra đặc trưng thời gian và tài khoản
assert modeling_df["hour"].between(0, 23).all(), "Giờ giao dịch không hợp lệ"
assert modeling_df["day_of_week"].between(1, 7).all(), "Thứ trong tuần không hợp lệ"
assert modeling_df["is_night_transaction"].isin([0, 1]).all(), "Cờ ban đêm không hợp lệ"
print("Kiểm tra đặc trưng cơ bản: ĐẠT")

Kiểm tra đặc trưng cơ bản: ĐẠT


In [86]:
# 4.10 - Chuẩn bị lịch sử đăng nhập theo trạng thái
# Chỉ lấy sự kiện trước giao dịch khi tính đặc trưng
failed_logins = logins[logins["status"] == "Failed"]
success_logins = logins[logins["status"] == "Success"]

failed_times = {
    key: np.sort(group["login_at"].to_numpy(dtype="datetime64[ns]"))
    for key, group in failed_logins.groupby("customer_key")
}
success_times = {
    key: np.sort(group["login_at"].to_numpy(dtype="datetime64[ns]"))
    for key, group in success_logins.groupby("customer_key")
}
print("Số đăng nhập thất bại:", len(failed_logins))
print("Số đăng nhập thành công:", len(success_logins))

Số đăng nhập thất bại: 8748
Số đăng nhập thành công: 27908


In [87]:
# 4.11 - Đếm đăng nhập trong 1 giờ và 24 giờ trước giao dịch
modeling_df["failed_login_count_1h"] = 0
modeling_df["failed_login_count_24h"] = 0
modeling_df["successful_login_count_24h"] = 0

for customer_key, group in modeling_df.groupby("customer_key"):
    transaction_times = group["transaction_at"].to_numpy(dtype="datetime64[ns]")

    failed = failed_times.get(customer_key)
    if failed is not None:
        end = np.searchsorted(failed, transaction_times, side="left")
        start_1h = np.searchsorted(failed, transaction_times - np.timedelta64(1, "h"), side="left")
        start_24h = np.searchsorted(failed, transaction_times - np.timedelta64(24, "h"), side="left")
        modeling_df.loc[group.index, "failed_login_count_1h"] = end - start_1h
        modeling_df.loc[group.index, "failed_login_count_24h"] = end - start_24h

    successful = success_times.get(customer_key)
    if successful is not None:
        end = np.searchsorted(successful, transaction_times, side="left")
        start = np.searchsorted(successful, transaction_times - np.timedelta64(24, "h"), side="left")
        modeling_df.loc[group.index, "successful_login_count_24h"] = end - start

assert (modeling_df["failed_login_count_1h"] <= modeling_df["failed_login_count_24h"]).all()
print("Đã tạo 3 đặc trưng lịch sử đăng nhập")

Đã tạo 3 đặc trưng lịch sử đăng nhập


In [88]:
# 4.12 - Chuẩn bị lịch sử giao dịch theo khách hàng
history = tx.sort_values(["customer_key", "transaction_at", "transaction_id"]).copy()
windows = {
    "txn_count_1h": np.timedelta64(1, "h"),
    "txn_count_24h": np.timedelta64(24, "h"),
    "txn_count_7d": np.timedelta64(7, "D"),
    "txn_count_30d": np.timedelta64(30, "D"),
}
for column in windows:
    history[column] = 0
for column in ["avg_amount_30d", "max_amount_30d", "time_since_last_transaction"]:
    history[column] = np.nan
print("Đã chuẩn bị lịch sử giao dịch")

Đã chuẩn bị lịch sử giao dịch


In [89]:
# 4.13 - Tính tần suất, số tiền và khoảng cách với giao dịch trước
for customer_key, group in history.groupby("customer_key"):
    times = group["transaction_at"].to_numpy(dtype="datetime64[ns]")
    amounts = group["amount"].to_numpy(dtype=float)

    # side="left" để không tính giao dịch hiện tại và giao dịch cùng thời điểm
    end = np.searchsorted(times, times, side="left")
    for column, window in windows.items():
        start = np.searchsorted(times, times - window, side="left")
        history.loc[group.index, column] = end - start

    start_30d = np.searchsorted(times, times - np.timedelta64(30, "D"), side="left")
    history.loc[group.index, "avg_amount_30d"] = [
        amounts[a:b].mean() if b > a else np.nan for a, b in zip(start_30d, end)
    ]
    history.loc[group.index, "max_amount_30d"] = [
        amounts[a:b].max() if b > a else np.nan for a, b in zip(start_30d, end)
    ]

    # Chưa có giao dịch trước thì giữ NaN, không thay bằng 0
    minutes = np.full(len(group), np.nan)
    has_previous = end > 0
    minutes[has_previous] = (
        (times[has_previous] - times[end[has_previous] - 1]) / np.timedelta64(1, "m")
    )
    history.loc[group.index, "time_since_last_transaction"] = minutes

history["amount_to_avg_ratio"] = (
    history["amount"] / history["avg_amount_30d"].replace(0, np.nan)
)
print("Đã tạo 8 đặc trưng lịch sử giao dịch")

Đã tạo 8 đặc trưng lịch sử giao dịch


In [90]:
# 4.14 - Kiểm tra và ghép đặc trưng lịch sử
assert (history["txn_count_1h"] <= history["txn_count_24h"]).all()
assert (history["txn_count_24h"] <= history["txn_count_7d"]).all()
assert (history["txn_count_7d"] <= history["txn_count_30d"]).all()

history_cols = list(windows) + [
    "avg_amount_30d", "max_amount_30d", "amount_to_avg_ratio", "time_since_last_transaction"
]
modeling_df = modeling_df.merge(
    history[["transaction_id"] + history_cols],
    on="transaction_id", how="left", validate="one_to_one"
)
assert len(modeling_df) == len(tx)
print("Ghép đặc trưng lịch sử: ĐẠT")

Ghép đặc trưng lịch sử: ĐẠT


In [91]:
# 4.15 - Chọn nhãn gian lận đã được xác nhận
labels = labels[labels["label_source"].eq("SIMULATED_GROUND_TRUTH")].copy()
assert labels["transaction_id"].is_unique
labels["is_fraud"] = labels["label"].map({"LEGITIMATE": 0, "FRAUD": 1})
assert labels["is_fraud"].isin([0, 1]).all()
print("Số nhãn hợp lệ:", len(labels))

Số nhãn hợp lệ: 30000


In [92]:

# 4.16 - Ghép nhãn gian lận với bảng giao dịch

modeling_df = modeling_df.drop(
    columns=[
        "is_fraud", "labeled_at",
        "is_fraud_x", "is_fraud_y",
        "labeled_at_x", "labeled_at_y"
    ],
    errors="ignore"
)

# Ghép nhãn gian lận đã được xác nhận
modeling_df = modeling_df.merge(
    labels[["transaction_id", "is_fraud", "labeled_at"]],
    on="transaction_id",
    how="left",
    validate="one_to_one"
)

# Kiểm tra kết quả
assert modeling_df["is_fraud"].isin([0, 1]).all(), "Nhãn gian lận không hợp lệ"
assert modeling_df["labeled_at"].notna().all(), "Thiếu thời điểm ghi nhận nhãn"

print("Số giao dịch theo nhãn:")
print(modeling_df["is_fraud"].value_counts())

Số giao dịch theo nhãn:
is_fraud
0    27124
1     2876
Name: count, dtype: int64


In [93]:
# 4.17 - Tạo và chọn 19 đặc trưng để huấn luyện

# Đổi tên cột channel
modeling_df = modeling_df.rename(
    columns={"channel_key": "channel"}
)

# Tạo feature mới: Tỷ lệ đăng nhập thất bại trong 24 giờ
modeling_df["login_failure_ratio_24h"] = (
    modeling_df["failed_login_count_24h"] /
    (
        modeling_df["failed_login_count_24h"] +
        modeling_df["successful_login_count_24h"] + 1
    )
)

# Danh sách 19 feature
final_features = [
    "amount", "channel", "type_code",
    "hour", "day_of_week", "is_night_transaction",
    "account_age_days",
    "txn_count_1h", "txn_count_24h",
    "txn_count_7d", "txn_count_30d",
    "avg_amount_30d", "max_amount_30d",
    "amount_to_avg_ratio",
    "failed_login_count_24h",
    "successful_login_count_24h",
    "failed_login_count_1h",
    "time_since_last_transaction",
    "login_failure_ratio_24h"
]

# Kiểm tra số lượng và tên feature
assert len(final_features) == 19
assert len(set(final_features)) == 19
assert set(final_features).issubset(modeling_df.columns), (
    "Thiếu đặc trưng"
)

print("Số đặc trưng:", len(final_features))


Số đặc trưng: 19


In [94]:
# 4.18 - Tạo bộ dữ liệu gồm 21 đặc trưng và 4 cột hỗ trợ

support_cols = [
    "transaction_id",
    "transaction_at",
    "is_fraud",
    "labeled_at"
]

export_df = modeling_df[
    ["transaction_id", "transaction_at"] +
    final_features +
    ["is_fraud", "labeled_at"]
].copy()

assert export_df.shape == (len(tx), 23)
assert export_df["transaction_id"].is_unique

assert not {
    "amount_log",
    "branch_age_days",
    "hour_sin",
    "hour_cos"
} & set(export_df.columns)

print("Kích thước bộ dữ liệu:", export_df.shape)


Kích thước bộ dữ liệu: (30000, 23)


In [95]:
# 4.19 - Kiểm tra dữ liệu trước khi lưu

# Các feature được phép có giá trị thiếu
allow_missing = [
    "avg_amount_30d",
    "max_amount_30d",
    "amount_to_avg_ratio",
    "time_since_last_transaction"
]

# Những cột bắt buộc có giá trị
required = [
    column for column in final_features
    if column not in allow_missing
]

# Kiểm tra dữ liệu bắt buộc không bị thiếu
assert export_df[required + support_cols].notna().all().all(), (
    "Thiếu dữ liệu bắt buộc"
)

# Kiểm tra nhãn gian lận
assert export_df["is_fraud"].isin([0, 1]).all(), (
    "Nhãn gian lận không hợp lệ"
)

# Kiểm tra giá trị vô cực
numeric_data = export_df[final_features].select_dtypes(
    include="number"
)

assert not np.isinf(
    numeric_data.to_numpy(dtype=float)
).any(), "Có giá trị vô cực"

# Kiểm tra feature mới: Tỷ lệ đăng nhập thất bại
assert export_df["login_failure_ratio_24h"].between(0, 1).all(), (
    "Tỷ lệ đăng nhập thất bại không hợp lệ"
)

# Kiểm tra số lượng feature và tổng số cột
assert len(final_features) == 19
assert export_df.shape[1] == 23

print("Kiểm tra dữ liệu: ĐẠT")

print("\nGiá trị thiếu hợp lệ:")
print(export_df[allow_missing].isna().sum())


Kiểm tra dữ liệu: ĐẠT

Giá trị thiếu hợp lệ:
avg_amount_30d                 2896
max_amount_30d                 2896
amount_to_avg_ratio            2896
time_since_last_transaction    1200
dtype: int64


In [96]:
# 4.20 - Lưu CSV đúng thư mục dự án
# Nếu dữ liệu nằm trong data/raw, suy ra thư mục gốc của dự án
if DATA_DIR.name == "raw" and DATA_DIR.parent.name == "data":
    PROJECT_DIR = DATA_DIR.parent.parent
elif DATA_DIR.name == "data":
    PROJECT_DIR = DATA_DIR.parent
else:
    PROJECT_DIR = Path.cwd().parent if Path.cwd().name.lower() == "notebooks" else Path.cwd()
output_dir = PROJECT_DIR / "data" / "processed"
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / "fraud_modeling_dataset.csv"
export_df.to_csv(output_path, index=False, encoding="utf-8-sig")
print("Đã lưu:", output_path.resolve())

Đã lưu: D:\banking-fraud-detection\data\processed\fraud_modeling_dataset.csv


In [98]:
# 4.21 - Đọc lại file CSV để kiểm tra
saved_df = pd.read_csv(output_path)
assert saved_df.shape == (len(tx), 23), "Số dòng hoặc số cột không đúng"
assert saved_df["transaction_id"].is_unique
assert set(final_features).issubset(saved_df.columns)
print("Số giao dịch:", len(saved_df))
print("Số đặc trưng:", len(final_features))
print("Tổng số cột CSV:", saved_df.shape[1])
print("Kiểm tra file CSV: ĐẠT")

Số giao dịch: 30000
Số đặc trưng: 19
Tổng số cột CSV: 23
Kiểm tra file CSV: ĐẠT
